# RAG-Augmented Agent — Fine-Tune Against a Knowledge Base

Most production agents *need to ground their answers in something* —
your docs, your help center, your product catalog. This notebook shows
the smallest viable version: a tiny BM25 retriever, a knowledge-base
JSONL, and a reward that **penalizes ungrounded claims**.

**Runtime:** ~75 min on Colab A100. **Cost:** ~$0.50.

**Pillar:** this is dialogue + retrieval. The RL signal teaches the
agent *when* to lean on retrieved context, and *not* to hallucinate when
the retriever returns nothing useful.


## 1. Pin + install


In [ ]:
import os
import subprocess

PINNED_COMMIT = 'f8e498b'
if not os.path.exists('/content/stateset-agents'):
    subprocess.check_call([
        'git', 'clone', '--quiet',
        'https://github.com/stateset/stateset-agents',
        '/content/stateset-agents'
    ])
subprocess.check_call(['git', '-C', '/content/stateset-agents', 'checkout', '--quiet', PINNED_COMMIT])
%cd /content/stateset-agents
%pip install --quiet -e '.[training]'
%pip install --quiet accelerate
print('Installed.')


In [ ]:
from stateset_agents.utils.reproducibility import set_all_seeds

SEED = 42
state = set_all_seeds(SEED)
print('Seeds applied:', state.to_dict())


In [ ]:
%pip install --quiet rank-bm25


## 2. The knowledge base (~10 docs across 4 topics)


In [ ]:
KB = [
    {'doc_id': 'shipping-1', 'topic': 'shipping',
     'text': 'Standard shipping takes 5-7 business days within the US. International orders take 10-15 days.'},
    {'doc_id': 'shipping-2', 'topic': 'shipping',
     'text': 'Expedited shipping is available for an extra $9.95 and arrives in 1-2 business days.'},
    {'doc_id': 'refunds-1', 'topic': 'refunds',
     'text': 'Refunds are processed within 5 business days of receiving the returned item.'},
    {'doc_id': 'refunds-2', 'topic': 'refunds',
     'text': 'Items must be returned within 30 days of delivery to qualify for a refund.'},
    {'doc_id': 'returns-1', 'topic': 'returns',
     'text': 'Return shipping is free for damaged items; otherwise a $4.95 label fee is deducted from the refund.'},
    {'doc_id': 'returns-2', 'topic': 'returns',
     'text': 'Generate a prepaid return label from the order detail page under My Orders.'},
    {'doc_id': 'sizing-1', 'topic': 'sizing',
     'text': 'Sizes run true to standard US measurements. Size up for half-sizes.'},
    {'doc_id': 'sizing-2', 'topic': 'sizing',
     'text': 'Exchanges for a different size ship free both ways within 30 days.'},
    {'doc_id': 'billing-1', 'topic': 'billing',
     'text': 'We accept Visa, Mastercard, Amex, and PayPal. No store credit at this time.'},
    {'doc_id': 'billing-2', 'topic': 'billing',
     'text': 'Subscriptions can be cancelled anytime; the next charge is suspended within 24 hours.'},
]
print(f'{len(KB)} docs across {len({d["topic"] for d in KB})} topics.')


## 3. BM25 retriever (3 lines)


In [ ]:
from rank_bm25 import BM25Okapi

_corpus = [d['text'].lower().split() for d in KB]
_bm25 = BM25Okapi(_corpus)

def retrieve(query: str, k: int = 2):
    tokens = query.lower().split()
    scores = _bm25.get_scores(tokens)
    ranked = sorted(zip(scores, KB), key=lambda x: -x[0])[:k]
    return [{'doc_id': d['doc_id'], 'text': d['text'], 'bm25': float(s)} for s, d in ranked]

for q in ['How long does shipping take?', 'I want to return my shoes']:
    print(q)
    for r in retrieve(q): print(f"  [{r['doc_id']}] {r['text'][:60]}... bm25={r['bm25']:.2f}")


## 4. Scenarios + grounding reward

The reward awards (a) acknowledging the question, (b) **citing at least
one retrieved doc by its id or substring**, (c) avoiding hallucinated
numbers. The citation requirement is the grounding signal — it's what
makes RAG-trained agents prefer 'According to <doc-1>...' over
'I think...'.


In [ ]:
from typing import Any
from stateset_agents.core.reward_base import RewardFunction, RewardResult, RewardType
from stateset_agents.core.trajectory import ConversationTurn

SCENARIOS = [
    {'user_query': 'How long does shipping take to the US?',  'topic': 'shipping',
     'expected_keywords': ['5-7', 'business days']},
    {'user_query': 'When will my refund show up?',           'topic': 'refunds',
     'expected_keywords': ['5 business days']},
    {'user_query': 'How do I get a return label?',           'topic': 'returns',
     'expected_keywords': ['return label', 'order detail']},
    {'user_query': 'Do you accept Apple Pay?',               'topic': 'billing',
     'expected_keywords': ['Visa', 'Mastercard', 'PayPal']},
    {'user_query': 'How long for international shipping?',   'topic': 'shipping',
     'expected_keywords': ['10-15']},
    {'user_query': 'How do sizes run?',                      'topic': 'sizing',
     'expected_keywords': ['true to', 'half-sizes']},
]
TRAIN, EVAL = SCENARIOS[:4], SCENARIOS[4:]

class GroundingReward(RewardFunction):
    name = 'grounding'
    def __init__(self):
        super().__init__(weight=1.0, reward_type=RewardType.IMMEDIATE, name=self.name)
    async def compute_reward(self, turns, context=None):
        ctx = context or {}
        text = (turns[-1].content if turns else '') or ''
        retrieved = ctx.get('retrieved_docs', [])
        cited = any(d['doc_id'] in text or d['text'][:25] in text for d in retrieved)
        kw_hits = sum(1 for k in ctx.get('expected_keywords', []) if k.lower() in text.lower())
        kw_total = max(len(ctx.get('expected_keywords', [])), 1)
        score = 0.4 * (1.0 if cited else 0.0) + 0.6 * (kw_hits / kw_total)
        return RewardResult(score=score, breakdown={'cited': float(cited), 'kw_hit_rate': kw_hits/kw_total})

print('GroundingReward ready.')


## 5. Prompt template + baseline


In [ ]:
from stateset_agents import MultiTurnAgent
from stateset_agents.core.agent_config import AgentConfig
import torch

MODEL = 'Qwen/Qwen2.5-0.5B-Instruct'

def build_prompt(query: str, docs: list):
    doc_block = '\n'.join(f'[{d["doc_id"]}] {d["text"]}' for d in docs)
    return (
        'You are a helpful agent. Answer the user using ONLY the retrieved docs below. '
        'Cite the doc ID in brackets when you use it.\n\n'
        f'Retrieved:\n{doc_block}\n\n'
        f'User: {query}\n\nAgent:'
    )

async def evaluate(agent, scenarios):
    reward = GroundingReward()
    scores, breakdowns = [], []
    for s in scenarios:
        docs = retrieve(s['user_query'])
        prompt = build_prompt(s['user_query'], docs)
        response = await agent.generate_response(prompt)
        ctx = {**s, 'retrieved_docs': docs}
        r = await reward.compute_reward([ConversationTurn(role='assistant', content=response)], context=ctx)
        scores.append(r.score); breakdowns.append({'response_head': response[:80], **r.breakdown})
    return sum(scores)/len(scores), breakdowns

baseline_agent = MultiTurnAgent(AgentConfig(
    model_name=MODEL, torch_dtype='bfloat16', attn_implementation='sdpa',
    do_sample=False, temperature=0.0))
await baseline_agent.initialize()
baseline_score, baseline_breakdowns = await evaluate(baseline_agent, EVAL)
print(f'Baseline grounding: {baseline_score:.3f}')
for b in baseline_breakdowns: print(' ', b)
del baseline_agent
torch.cuda.empty_cache()


## 6. Train


In [ ]:
from stateset_agents.core import ConversationEnvironment
from stateset_agents.training import GSPOConfig, train_with_gspo
import time

config = GSPOConfig(
    model_name=MODEL, output_dir='/content/gspo_rag', report_to='none',
    num_generations=4,
    clip_range_left=3e-4, clip_range_right=4e-4,
    learning_rate=5e-6,
    max_prompt_length=768, max_completion_length=320,
    use_lora=True, lora_r=16, lora_alpha=32,
    num_epochs=1, warmup_ratio=0.1,
    use_reference_model=True, beta=0.05,
)
agent = MultiTurnAgent(AgentConfig(model_name=MODEL, torch_dtype='bfloat16', attn_implementation='sdpa'))
env = ConversationEnvironment(
    scenarios=[{**s, 'id': f'rag-{i}', 'topic': s['topic']} for i, s in enumerate(TRAIN)],
    reward_fn=GroundingReward(), max_turns=1)
queries = [
    {'prompt': build_prompt(s['user_query'], retrieve(s['user_query'])),
     'context': {**s, 'retrieved_docs': retrieve(s['user_query'])}}
    for s in TRAIN
]

t0 = time.time()
await train_with_gspo(config=config, agent=agent, environment=env,
                      reward_model=env.reward_fn, train_queries=queries)
print(f'Training wall-clock: {time.time()-t0:.0f}s')


## 7. Post-training eval + provenance


In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import json

final_score, final_breakdowns = await evaluate(agent, EVAL)
print(f'Final grounding: {final_score:.3f}  (Δ {final_score - baseline_score:+.3f})')
for b in final_breakdowns: print(' ', b)

result = {
    'trainer': 'gspo', 'task': 'rag_grounding',
    'model': MODEL, 'seed': SEED, 'pinned_commit': PINNED_COMMIT,
    'baseline_score': baseline_score, 'final_score': final_score,
    'delta': final_score - baseline_score,
    'kb_size': len(KB), 'n_train': len(TRAIN), 'n_eval': len(EVAL),
    'timestamp_utc': datetime.now(timezone.utc).isoformat(),
}
Path('/content/rag_result.json').write_text(json.dumps(result, indent=2))
print('Wrote /content/rag_result.json')


## 8. Next steps

- **Bigger KB.** 10 docs is a smoke test. Swap BM25 for a real retriever
  (FAISS + a small embedding model) once you have 1k+ docs.
- **Negative samples.** Train on scenarios where the retriever fails —
  the agent should learn to say 'I don't know' rather than hallucinate.
- **Hybrid retrieval.** BM25 + dense + a reranker; the reward is
  retrieval-independent so the same loop works for any of them.
